In [1]:
#!/usr/bin/env python3
"""
SPLADE Word-Level Workflow - Simple Jupyter Notebook

This notebook imports and uses the existing quick_builder.py and quick_evaluation.py
functions directly. Much cleaner than rewriting everything!
"""

import sys
import time
import json
from pathlib import Path
from typing import Dict, Any, Optional

# Add project root to path
project_root = Path('/work')
sys.path.append(str(project_root))

# Import the existing modules and their main functions
from sandbox.splade.indexing.quick_builder import SpladeIndexBuilder
from sandbox.splade.evaluation.quick_evaluation import main_splade_evaluation
import pickle
import torch

class PicklableSpladeVectorizer:
    """SPLADE vectorizer that can be pickled."""
    
    def __init__(self, vocabulary, vocab_path, base_model):
        # Store references instead of the full objects
        self.vocabulary_ = vocabulary
        self.feature_names_ = list(vocabulary.keys())
        # We'll recreate the encoder when needed
        self._vocab_path = vocab_path
        self._base_model = base_model
    
    def transform(self, texts):
        """Transform texts to sparse SPLADE representations."""
        # Lazy load the encoder when needed
        if not hasattr(self, '_encoder'):
            from sandbox.splade.core.models import create_word_splade_system
            _, _, self._encoder = create_word_splade_system(
                vocab_path=Path(self._vocab_path),
                base_model_name=self._base_model,
                device="cuda" if torch.cuda.is_available() else "cpu"
            )
        
        if isinstance(texts, str):
            texts = [texts]
        
        # Generate sparse representations
        sparse_vectors = self._encoder.encode(
            texts=texts,
            batch_size=min(32, len(texts)),
            max_length=512,
            apply_sparsity=True
        )
        
        # Convert to sparse matrix format expected by evaluation code
        from scipy.sparse import csr_matrix
        sparse_np = sparse_vectors.cpu().numpy()
        return csr_matrix(sparse_np)
    
    def get_feature_names_out(self):
        """Get feature names (words)."""
        import numpy as np
        return np.array(self.feature_names_)

def save_index_fixed(builder, index_data: Dict, output_path: Path, collection_name: str):
    """
    Fixed save_index function that handles the pickling issue.
    """
    print(f"💾 Saving {collection_name} index...")
    
    # Create output directory
    index_dir = output_path / f"{collection_name}_splade_quick_index"
    index_dir.mkdir(parents=True, exist_ok=True)
    
    # Save documents (compatible with evaluation code)
    with open(index_dir / "documents.pkl", "wb") as f:
        pickle.dump(index_data['documents'], f)
    
    # Save embeddings as sparse matrix (compatible with evaluation code)
    from scipy.sparse import save_npz
    save_npz(index_dir / "embeddings.npz", index_data['sparse_matrix'])
    
    # Save metadata
    with open(index_dir / "metadata.json", "w", encoding='utf-8') as f:
        json.dump(index_data['metadata'], f, indent=2, ensure_ascii=False)
    
    # Save item keys for quick lookup
    with open(index_dir / "item_keys.json", "w", encoding='utf-8') as f:
        json.dump(index_data['item_keys'], f, indent=2, ensure_ascii=False)
    
    # Create the picklable vectorizer
    vectorizer = PicklableSpladeVectorizer(
        vocabulary=index_data['vectorizer'].vocabulary_,
        vocab_path="/work/sandbox/splade/vocabularies/OEB_vocab_filtered_50000",
        base_model="dccuchile/bert-base-spanish-wwm-cased"
    )
    
    # Save vectorizer (compatible with evaluation code)
    with open(index_dir / "vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)
    
    print(f"📁 Index saved to: {index_dir}")
    print(f"📋 Files created:")
    print(f"  - documents.pkl (documents)")
    print(f"  - vectorizer.pkl (SPLADE vectorizer)")
    print(f"  - embeddings.npz (sparse representations)")
    print(f"  - metadata.json (index metadata)")
    print(f"  - item_keys.json (document keys)")
    
    return index_dir

def quick_test_workflow(max_docs: int = 1000, sample_size: int = 500):
    """
    Run a quick test of the SPLADE workflow with limited documents.
    
    Args:
        max_docs: Maximum documents per collection
        sample_size: Sample size for evaluation
    """
    print("🧪 SPLADE QUICK TEST WORKFLOW")
    print("="*50)
    print(f"📊 Processing {max_docs:,} documents per collection")
    print(f"📦 Evaluation sample: {sample_size:,} queries")
    
    workflow_start = time.time()
    
    try:
        # Initialize the SPLADE builder
        print("\n🔧 Initializing SPLADE Index Builder...")
        builder = SpladeIndexBuilder(
            vocab_path=Path("/work/sandbox/splade/vocabularies"),
            use_filtered=True,  # Use filtered vocabulary
            device=None  # Auto-detect
        )
        
        # Build texto index
        print("\n📚 Building TEXTO index...")
        texto_docs = builder.load_documents("OEB", "texto")
        texto_index_data = builder.create_splade_vectorizer(texto_docs, max_docs=max_docs)
        texto_index_dir = save_index_fixed(
            builder,
            texto_index_data, 
            Path("/work/sandbox/splade/indices"), 
            "OEB_texto"
        )
        
        # Build resumen index  
        print("\n📄 Building RESUMEN index...")
        resumen_docs = builder.load_documents("OEB", "resumen")
        resumen_index_data = builder.create_splade_vectorizer(resumen_docs, max_docs=max_docs)
        resumen_index_dir = save_index_fixed(
            builder,
            resumen_index_data,
            Path("/work/sandbox/splade/indices"),
            "OEB_resumen"
        )
        
        # Analyze the indices
        print("\n📊 Analyzing TEXTO index...")
        builder.analyze_splade_index(texto_index_data)
        
        print("\n📊 Analyzing RESUMEN index...")  
        builder.analyze_splade_index(resumen_index_data)
        
        # Run evaluation
        print("\n🔍 Running SPLADE evaluation...")
        results_data = main_splade_evaluation(
            texto_index_path=texto_index_dir,
            resumen_docs=resumen_docs,
            file_identifier="quick_test",
            batch_size=500,
            top_k=10,
            sample_size=sample_size
        )
        
        # Print summary
        workflow_elapsed = time.time() - workflow_start
        workflow_min = int(workflow_elapsed // 60)
        workflow_sec = int(workflow_elapsed % 60)
        
        print(f"\n✅ QUICK TEST COMPLETED SUCCESSFULLY!")
        print(f"⏱️  Total time: {workflow_min}m {workflow_sec}s")
        
        return True, results_data
        
    except Exception as e:
        print(f"❌ Quick test failed: {e}")
        import traceback
        traceback.print_exc()
        return False, None

def production_workflow(sample_size: Optional[int] = None):
    """
    Run the production SPLADE workflow with all documents.
    
    Args:
        sample_size: Sample size for evaluation (None for default)
    """
    print("🏭 SPLADE PRODUCTION WORKFLOW")
    print("="*50)
    print("📊 Processing ALL documents")
    
    workflow_start = time.time()
    
    try:
        # Initialize the SPLADE builder
        print("\n🔧 Initializing SPLADE Index Builder...")
        builder = SpladeIndexBuilder(
            vocab_path=Path("/work/sandbox/splade/vocabularies"),
            use_filtered=True,
            device=None
        )
        
        # Build texto index
        print("\n📚 Building TEXTO index (all documents)...")
        texto_docs = builder.load_documents("OEB", "texto")
        texto_index_data = builder.create_splade_vectorizer(texto_docs, max_docs=None)
        texto_index_dir = builder.save_index(
            texto_index_data,
            Path("/work/sandbox/splade/indices"),
            "OEB_texto"
        )
        
        # Build resumen index
        print("\n📄 Building RESUMEN index (all documents)...")
        resumen_docs = builder.load_documents("OEB", "resumen")
        resumen_index_data = builder.create_splade_vectorizer(resumen_docs, max_docs=None)
        resumen_index_dir = builder.save_index(
            resumen_index_data,
            Path("/work/sandbox/splade/indices"),
            "OEB_resumen"
        )
        
        # Analyze the indices
        print("\n📊 Analyzing indices...")
        builder.analyze_splade_index(texto_index_data)
        builder.analyze_splade_index(resumen_index_data)
        
        # Run evaluation
        print("\n🔍 Running SPLADE evaluation...")
        results_data = main_splade_evaluation(
            texto_index_path=texto_index_dir,
            resumen_index_path=resumen_index_dir,
            file_identifier="production",
            batch_size=500,
            top_k=10,
            sample_size=sample_size
        )
        
        # Print summary
        workflow_elapsed = time.time() - workflow_start
        workflow_min = int(workflow_elapsed // 60)
        workflow_sec = int(workflow_elapsed % 60)
        
        print(f"\n✅ PRODUCTION WORKFLOW COMPLETED!")
        print(f"⏱️  Total time: {workflow_min}m {workflow_sec}s")
        
        return True, results_data
        
    except Exception as e:
        print(f"❌ Production workflow failed: {e}")
        import traceback
        traceback.print_exc()
        return False, None

def custom_workflow(max_docs: Optional[int] = None,
                   sample_size: Optional[int] = None,
                   top_k: int = 10,
                   use_filtered: bool = True,
                   file_base: str = "OEB"):
    """
    Run custom SPLADE workflow with specified parameters.
    
    Args:
        max_docs: Maximum documents per collection (None for all)
        sample_size: Sample size for evaluation 
        top_k: Number of top results
        use_filtered: Use filtered vocabulary
        file_base: Base filename
    """
    print("⚙️ SPLADE CUSTOM WORKFLOW")
    print("="*50)
    print(f"📁 File base: {file_base}")
    print(f"📊 Max documents: {max_docs or 'All'}")
    print(f"🔍 Vocabulary: {'Filtered' if use_filtered else 'Full'}")
    print(f"🎯 Top-k: {top_k}")
    print(f"📦 Sample size: {sample_size or 'Default'}")
    
    workflow_start = time.time()
    
    try:
        # Initialize builder
        builder = SpladeIndexBuilder(
            vocab_path=Path("/work/sandbox/splade/vocabularies"),
            use_filtered=use_filtered,
            device=None
        )
        
        # Build indices
        print(f"\n📚 Building TEXTO index...")
        texto_docs = builder.load_documents(file_base, "texto")
        texto_index_data = builder.create_splade_vectorizer(texto_docs, max_docs=max_docs)
        texto_index_dir = builder.save_index(
            texto_index_data,
            Path("/work/sandbox/splade/indices"),
            f"{file_base}_texto"
        )
        
        print(f"\n📄 Building RESUMEN index...")
        resumen_docs = builder.load_documents(file_base, "resumen")
        resumen_index_data = builder.create_splade_vectorizer(resumen_docs, max_docs=max_docs)
        resumen_index_dir = builder.save_index(
            resumen_index_data,
            Path("/work/sandbox/splade/indices"),
            f"{file_base}_resumen"
        )
        
        # Run evaluation
        print(f"\n🔍 Running evaluation...")
        results_data = main_splade_evaluation(
            texto_index_path=texto_index_dir,
            resumen_index_path=resumen_index_dir,
            file_identifier="custom",
            batch_size=500,
            top_k=top_k,
            sample_size=sample_size
        )
        
        workflow_elapsed = time.time() - workflow_start
        workflow_min = int(workflow_elapsed // 60)
        workflow_sec = int(workflow_elapsed % 60)
        
        print(f"\n✅ CUSTOM WORKFLOW COMPLETED!")
        print(f"⏱️  Total time: {workflow_min}m {workflow_sec}s")
        
        return True, results_data
        
    except Exception as e:
        print(f"❌ Custom workflow failed: {e}")
        import traceback
        traceback.print_exc()
        return False, None

def evaluate_existing_indices(texto_index_path: str,
                             resumen_index_path: str,
                             sample_size: Optional[int] = None,
                             top_k: int = 10):
    """
    Evaluate existing SPLADE indices.
    
    Args:
        texto_index_path: Path to texto index
        resumen_index_path: Path to resumen index
        sample_size: Sample size for evaluation
        top_k: Number of top results
    """
    print("🔍 EVALUATING EXISTING SPLADE INDICES")
    print("="*50)
    print(f"📁 Texto index: {texto_index_path}")
    print(f"📁 Resumen index: {resumen_index_path}")
    
    try:
        results_data = main_splade_evaluation(
            texto_index_path=Path(texto_index_path),
            resumen_index_path=Path(resumen_index_path),
            file_identifier="existing",
            batch_size=500,
            top_k=top_k,
            sample_size=sample_size
        )
        
        print(f"\n✅ Evaluation completed!")
        return True, results_data
        
    except Exception as e:
        print(f"❌ Evaluation failed: {e}")
        import traceback
        traceback.print_exc()
        return False, None

# ============================================================================
# CONVENIENCE FUNCTIONS FOR JUPYTER
# ============================================================================

def run_quick_test():
    """Run quick test - paste this in a Jupyter cell"""
    success, results = quick_test_workflow(max_docs=1000, sample_size=500)
    return success, results

def run_production():
    """Run production workflow - paste this in a Jupyter cell"""
    success, results = production_workflow(sample_size=16590)
    return success, results

def run_custom(max_docs=None, sample_size=None, top_k=10):
    """Run custom workflow - paste this in a Jupyter cell"""
    success, results = custom_workflow(
        max_docs=max_docs,
        sample_size=sample_size,
        top_k=top_k
    )
    return success, results

def just_evaluate(texto_index, resumen_index, sample_size=None):
    """Just evaluate existing indices - paste this in a Jupyter cell"""
    success, results = evaluate_existing_indices(
        texto_index_path=texto_index,
        resumen_index_path=resumen_index,
        sample_size=sample_size
    )
    return success, results

# ============================================================================
# USAGE
# ============================================================================

print("🎯 SPLADE Workflow Notebook Ready!")
print("\n📝 Available functions:")
print("  run_quick_test()     - Quick test with 1K docs")
print("  run_production()     - Production run with all docs") 
print("  run_custom(...)      - Custom parameters")
print("  just_evaluate(...)   - Evaluate existing indices")
print("\n🚀 Example: success, results = run_quick_test()")

"""
USAGE EXAMPLES:

# Quick test
success, results = run_quick_test()

# Production run  
success, results = run_production()

# Custom run
success, results = run_custom(max_docs=5000, sample_size=1000, top_k=20)

# Just evaluate existing indices
success, results = just_evaluate(
    texto_index="/path/to/texto/index",
    resumen_index="/path/to/resumen/index",
    sample_size=1000
)
"""

🎯 SPLADE Workflow Notebook Ready!

📝 Available functions:
  run_quick_test()     - Quick test with 1K docs
  run_production()     - Production run with all docs
  run_custom(...)      - Custom parameters
  just_evaluate(...)   - Evaluate existing indices

🚀 Example: success, results = run_quick_test()


'\nUSAGE EXAMPLES:\n\n# Quick test\nsuccess, results = run_quick_test()\n\n# Production run  \nsuccess, results = run_production()\n\n# Custom run\nsuccess, results = run_custom(max_docs=5000, sample_size=1000, top_k=20)\n\n# Just evaluate existing indices\nsuccess, results = just_evaluate(\n    texto_index="/path/to/texto/index",\n    resumen_index="/path/to/resumen/index",\n    sample_size=1000\n)\n'

In [2]:
success, results = run_quick_test()

🧪 SPLADE QUICK TEST WORKFLOW
📊 Processing 1,000 documents per collection
📦 Evaluation sample: 500 queries

🔧 Initializing SPLADE Index Builder...
🚀 Initializing SPLADE Index Builder
📁 Vocabulary path: /work/sandbox/splade/vocabularies
🔍 Using filtered vocabulary
🧠 Base model: dccuchile/bert-base-spanish-wwm-cased
🖥️  Device: cuda
🔧 Creating Word-Level SPLADE System...
📚 Loading vocabulary...
Vocabulary loaded from /work/sandbox/splade/vocabularies/OEB_vocab_filtered_50000
Vocabulary size: 333
🔤 Creating tokenizer...
🧠 Creating model...


Some weights of the model checkpoint at dccuchile/bert-base-spanish-wwm-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.we

⚡ Creating encoder...
WordSpladeEncoder initialized on cuda
Vocabulary size: 333
✅ Word-Level SPLADE system ready!
📊 Vocabulary size: 333
🖥️  Device: cuda
✅ SPLADE system loaded successfully!
📊 Vocabulary size: 333

📚 Building TEXTO index...
📚 Loading texto documents with base name: OEB
📖 Loaded 47,514 texto documents
🔨 Creating SPLADE vectorizer...
📊 Processing 1,000 documents (limited)
📝 Extracted 1,000 valid texts
⚡ Generating SPLADE representations...


Processing batches:   0%|          | 0/32 [00:00<?, ?it/s]

🎯 Generated sparse matrix: (1000, 333)
📈 Average sparsity: 0.555
💾 Saving OEB_texto index...
📁 Index saved to: /work/sandbox/splade/indices/OEB_texto_splade_quick_index
📋 Files created:
  - documents.pkl (documents)
  - vectorizer.pkl (SPLADE vectorizer)
  - embeddings.npz (sparse representations)
  - metadata.json (index metadata)
  - item_keys.json (document keys)

📄 Building RESUMEN index...
📚 Loading resumen documents with base name: OEB
📖 Loaded 47,514 resumen documents
🔨 Creating SPLADE vectorizer...
📊 Processing 1,000 documents (limited)
📝 Extracted 1,000 valid texts
⚡ Generating SPLADE representations...


Processing batches:   0%|          | 0/32 [00:00<?, ?it/s]

🎯 Generated sparse matrix: (1000, 333)
📈 Average sparsity: 0.584
💾 Saving OEB_resumen index...
📁 Index saved to: /work/sandbox/splade/indices/OEB_resumen_splade_quick_index
📋 Files created:
  - documents.pkl (documents)
  - vectorizer.pkl (SPLADE vectorizer)
  - embeddings.npz (sparse representations)
  - metadata.json (index metadata)
  - item_keys.json (document keys)

📊 Analyzing TEXTO index...

SPLADE INDEX ANALYSIS
📊 Index statistics:
  Documents: 1,000
  Vocabulary size: 333
  Non-zero elements: 184,682
  Density: 0.554601 (55.4601%)
  Avg non-zero per doc: 184.7

📈 Value statistics:
  Max value: 5.2212
  Min non-zero: 0.000002

🔥 Most frequently activated words:
   1. 0,80 a 1,00 m        (1000 docs, 100.0%)
   2. ataque               (1000 docs, 100.0%)
   3. interfonía           (1000 docs, 100.0%)
   4. embocaduras          (1000 docs, 100.0%)
   5. telecomunicaciones   (1000 docs, 100.0%)
   6. preparación          (1000 docs, 100.0%)
   7. ancho                (1000 docs, 1

Some weights of the model checkpoint at dccuchile/bert-base-spanish-wwm-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.we

⚡ Creating encoder...
WordSpladeEncoder initialized on cuda
Vocabulary size: 333
✅ Word-Level SPLADE system ready!
📊 Vocabulary size: 333
🖥️  Device: cuda
📈 Batch 1/1 processed. Estimated time to completion: 0m 0s
⏱️  Performing overall analysis...

Overall Analysis Results:
  Recall: 0.0000
  Precision: 0.0000
  F1 Score: 0.0000
  Mean Reciprocal Rank (MRR): 0.0000
  Mean Average Precision (MAP): 0.0000
✅ Performing overall analysis completed in 0m 0s
⏱️  Performing single-match analysis...

Single-Match Analysis Results:
Success@10             : 0.00%
MRR: 0.0000
Mean Position: nan
Median Position: nan
Perfect Matches: 0.00%

Position Distribution:
------------------------------
Position_1: 0.00%
Position_2: 0.00%
Position_3: 0.00%
Position_4: 0.00%
Position_5: 0.00%
Position_6: 0.00%
Position_7: 0.00%
Position_8: 0.00%
Position_9: 0.00%
Position_10: 0.00%
✅ Performing single-match analysis completed in 0m 0s
⏱️  Performing prefix-matching analysis...

Prefix Matching Analysis Result

/opt/conda/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
Traceback (most recent call last):
  File "/tmp/ipykernel_23777/3446869797.py", line 170, in quick_test_workflow
    results_data = main_splade_evaluation(
                   ^^^^^^^^^^^^^^^^^^^^^^^
  File "/work/sandbox/splade/evaluation/quick_evaluation.py", line 428, in main_splade_evaluation
    'resumen_docs_path': str(resumen_docs_path),
                             ^^^^^^^^^^^^^^^^^
NameError: name 'resumen_docs_path' is not defined
